In [1]:
import os
import pandas as pd

def extract_means_from_test_folder(folder_path):
    """提取 makespan 和 emission 的均值，如果文件缺失或无有效值，返回 'NA'。"""
    ms_mean, em_mean = 'NA', 'NA'
    try:
        ms_file = next(f for f in os.listdir(folder_path) if f.startswith("makespan") and f.endswith(".xlsx"))
        ms_df = pd.read_excel(os.path.join(folder_path, ms_file))
        ms_values = pd.to_numeric(ms_df.iloc[:, 1], errors='coerce').dropna()
        if not ms_values.empty:
            ms_mean = ms_values.mean()
    except:
        pass

    try:
        em_file = next(f for f in os.listdir(folder_path) if f.startswith("emission") and f.endswith(".xlsx"))
        em_df = pd.read_excel(os.path.join(folder_path, em_file))
        em_values = pd.to_numeric(em_df.iloc[:, 1], errors='coerce').dropna()
        if not em_values.empty:
            em_mean = em_values.mean()
    except:
        pass

    return ms_mean, em_mean

# def find_subfolder(base_path, keyword):
#     # """查找包含 keyword 的子文件夹"""
#     # try:
#     #     return next(f for f in os.listdir(base_path) if keyword in f and os.path.isdir(os.path.join(base_path, f)))
#     # except:
#     #     return None
def find_subfolder(base_path, prefix):
    """查找以 prefix 开头的子文件夹名（例如 test_gnn 或 test_llm）"""
    try:
        return next(f for f in os.listdir(base_path) if f.startswith(prefix) and os.path.isdir(os.path.join(base_path, f)))
    except:
        return None

def analyze_all_em_folders(save_path):
    results = []
    folder_list = sorted([f for f in os.listdir(save_path) if f.startswith("em_") and os.path.isdir(os.path.join(save_path, f))])

    for idx, folder in enumerate(folder_list, start=1):
        folder_path = os.path.join(save_path, folder)
        gnn_folder = find_subfolder(folder_path, "test_gnn")
        llm_folder = find_subfolder(folder_path, "test_llm")

        gnn_ms, gnn_em = extract_means_from_test_folder(os.path.join(folder_path, gnn_folder)) if gnn_folder else ('NA', 'NA')
        llm_ms, llm_em = extract_means_from_test_folder(os.path.join(folder_path, llm_folder)) if llm_folder else ('NA', 'NA')

        try:
            ms_imp = (gnn_ms - llm_ms) / llm_ms * 100
        except:
            ms_imp = 'NA'

        try:
            em_imp = (gnn_em - llm_em) / llm_em * 100
        except:
            em_imp = 'NA'

        results.append({
            'index': idx,
            'folder': folder,
            'llm_ms_mean': llm_ms,
            'gnn_ms_mean': gnn_ms,
            'llm_em_mean': llm_em,
            'gnn_em_mean': gnn_em,
            'ms_improvement(%)': ms_imp,
            'em_improvement(%)': em_imp,
        })

    return pd.DataFrame(results)

# 示例调用
# save_dir = '/Users/yourname/Desktop/FINAL/0716_sample10/save'
# df = analyze_all_em_folders(save_dir)
# df.to_excel("summary_results.xlsx", index=False)

In [2]:
save_path='/home/zhiying/desktop/FINAL/0716_sample10/save'
df = analyze_all_em_folders(save_path)
# df.to_excel("summary_results.xlsx", index=False)
df

,index,folder,llm_ms_mean,gnn_ms_mean,llm_em_mean,gnn_em_mean,ms_improvement(%),em_improvement(%)
0,1,em_0.2_0.9_1.3_1.8_3.2_lambda_0_5,110.17,114.12,736.822985,730.614001,3.585368,-0.84267
1,2,em_0.2_3.2_2.8_1.6_0.9_lambda_0_5,109.64,115.1,863.840019,855.595015,4.979934,-0.95446
2,3,em_0.3_1.2_2.1_3.5_4.8_lambda_0_5,111.66,115.01,1177.827014,1173.258992,3.000179,-0.387835
3,4,em_0.3_2.0_4.9_0.7_4.8_lambda_0_5,110.75,114.17,1248.144006,1232.780992,3.088036,-1.230869
4,5,em_0.3_3.0_4.7_4.9_1.6_lambda_0_5,111.34,114.22,1441.134978,1430.828956,2.586671,-0.715132
5,6,em_0.4_1.7_0.3_4.5_4.8_lambda_0_5,115.35,112.66,1140.04301,1148.336008,-2.332033,0.727428
6,7,em_0.8_3.2_0.2_1.4_2.4_lambda_0_5,112.3,114.6,792.360005,789.828005,2.048085,-0.319552
7,8,em_16.0_3.1_2.3_0.9_5.0_lambda_0_5,113.83,112.25,2628.802042,2638.130044,-1.388035,0.354838
8,9,em_16_0.3_1.2_3_0.9_5_lambda_0_5,NA,NA,NA,NA,NA,NA
9,10,em_3.2_2.0_2.8_0.2_1.1_lambda_0_5,112.4,111.75,921.678998,923.328991,-0.578292,0.17902
